In [28]:
import pandas as pd
import numpy as np

In [29]:
#1. Đọc file csv
df = pd.read_csv("../data/extracted/muaban_raw.csv")

In [30]:
# 2. Bỏ dòng thiếu ít nhất 1 cột ban đầu
df = df.dropna(how='any')

In [31]:
print(len(df))

1782


In [32]:
# Xem 5 dòng đầu
print(df.head())

# Kiểu dữ liệu ban đầu của mỗi cột
print(df.dtypes)

                                              tieu_de       gia  \
3        Bán nhà 6 tầng ngõ ô tô đường Hoàng Hoa Thám     42 tỷ   
10  Bán nhà 6t MP Nguyễn Thái Học DT 80m2 MT4,2m s...     35 tỷ   
11  Bán nhà 8t Thang Máy MP Nguyễn Thái Học- Điện ...  89,99 tỷ   
13  Bán nhà nhỏ phố Sơn Tây-Ba Đình 26m/2 tầng .Gí...    5,7 tỷ   
14  BÁN BT KHU PL Vip Giảng Võ-  BA ĐÌNH ,OTO TRÁN...     66 tỷ   

                                              dia_chi      dien_tich_dat  \
3   Đường Hoàng Hoa Thám, Phường Liễu Giai, Quận B...  140 m² (7.0x21.0)   
10  Đường Nguyễn Thái Học, Phường Điện Biên, Quận ...              80 m²   
11  68, Đường Nguyễn Thái Học, Phường Điện Biên, Q...              92 m²   
13  Đường Sơn Tây, Phường Kim Mã, Quận Ba Đình, Hà...    26 m² (3.2x8.0)   
14  66, Đường Giảng Võ, Phường Giảng Võ, Quận Ba Đ...           136.8 m²   

   phong_ngu phong_tam  so_tang phap_ly   ngay_dang  
3    9 phòng     11 WC      6.0   Sổ đỏ  25-11-2025  
10   6 phòng      7 WC      7.0 

In [33]:
# 3. Chuẩn hóa cột Giá (gia) về float (tỷ đồng)
df['gia'] = df['gia'].str.replace(r'[^\d,]', '', regex=True)   # giữ số và dấu ,
df['gia'] = df['gia'].str.replace(',', '.', regex=False)       # đổi , -> .
df['gia'] = pd.to_numeric(df['gia'], errors='coerce')          # không chuyển được -> NaN

# Bỏ dòng có giá NaN
df = df.dropna(subset=['gia'])

# 4. Chuẩn hóa diện tích đất về float (m²)
df['dien_tich_dat'] = df['dien_tich_dat'].str.extract(r'([\d,.]+)')[0]  # lấy phần số
df['dien_tich_dat'] = df['dien_tich_dat'].str.replace(',', '.', regex=False).astype(float)

# 5. Phòng ngủ, phòng tắm về số nguyên
df['phong_ngu'] = pd.to_numeric(df['phong_ngu'].str.extract(r'(\d+)')[0], errors='coerce').astype('Int64')
df['phong_tam'] = pd.to_numeric(df['phong_tam'].str.extract(r'(\d+)')[0], errors='coerce').astype('Int64')

# 6. Số tầng về số nguyên, làm tròn
df['so_tang'] = pd.to_numeric(df['so_tang'], errors='coerce').round().astype('Int64')

# 7. Loại bỏ các dòng có giá trị số âm
df = df[(df['gia'] >= 0) & (df['dien_tich_dat'] >= 0)]
df = df[(df['phong_ngu'].fillna(0) >= 0) & (df['phong_tam'].fillna(0) >= 0) & (df['so_tang'] >= 0)]

# 8. Chuẩn hóa pháp lý
def chuan_hoa_phap_ly(x):
    x_lower = str(x).lower()
    if "sổ đỏ" in x_lower and "sổ hồng" in x_lower:
        return "Sổ hồng"
    elif "sổ đỏ" in x_lower:
        return "Sổ đỏ"
    elif "sổ hồng" in x_lower:
        return "Sổ hồng"
    else:
        return "Giấy tờ không xác định"

df['phap_ly'] = df['phap_ly'].apply(chuan_hoa_phap_ly)

# 9. Chuẩn hóa ngày đăng về datetime
df['ngay_dang'] = pd.to_datetime(df['ngay_dang'], dayfirst=True, errors='coerce')

# 10. Bỏ trùng lặp theo tieu_de + dia_chi
df = df.drop_duplicates(subset=['tieu_de', 'dia_chi'], keep='first')

# 11. Bỏ dòng thiếu sau khi chuẩn hóa
df = df.dropna()

# 12. Kiểm tra kết quả
print(df.info())
print(df.head())

<class 'pandas.core.frame.DataFrame'>
Index: 1759 entries, 3 to 2519
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   tieu_de        1759 non-null   object        
 1   gia            1759 non-null   float64       
 2   dia_chi        1759 non-null   object        
 3   dien_tich_dat  1759 non-null   float64       
 4   phong_ngu      1759 non-null   Int64         
 5   phong_tam      1759 non-null   Int64         
 6   so_tang        1759 non-null   Int64         
 7   phap_ly        1759 non-null   object        
 8   ngay_dang      1759 non-null   datetime64[ns]
dtypes: Int64(3), datetime64[ns](1), float64(2), object(3)
memory usage: 142.6+ KB
None
                                              tieu_de    gia  \
3        Bán nhà 6 tầng ngõ ô tô đường Hoàng Hoa Thám  42.00   
10  Bán nhà 6t MP Nguyễn Thái Học DT 80m2 MT4,2m s...  35.00   
11  Bán nhà 8t Thang Máy MP Nguyễn Thái Học- Điện ...  89.

In [34]:
# Lưu file CSV đã preprocess
df.to_csv("../data/preprocessed/muaban_preprocessed.csv", index=False, encoding='utf-8-sig')